# ⏩ Notebook 02 — Offset & Navigation Functions

**Functions covered:** `LAG()`, `LEAD()`, `FIRST_VALUE()`, `LAST_VALUE()`, `NTH_VALUE()`

## Quick Reference

| Function | Description |
|----------|-------------|
| `LAG(col, n, default)` | Value from n rows **before** current row |
| `LEAD(col, n, default)` | Value from n rows **after** current row |
| `FIRST_VALUE(col)` | First value in the window frame |
| `LAST_VALUE(col)` | Last value in the window frame (needs full frame!) |
| `NTH_VALUE(col, n)` | nth value in the window frame |

> ⚠️ `LAST_VALUE` requires `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING` to see the true last row.


In [ ]:
import duckdb, pandas as pd

employees = pd.read_csv("../data/employees.csv")
sales     = pd.read_csv("../data/sales.csv")
con = duckdb.connect()
con.register("employees", employees)
con.register("sales",     sales)

def sql(query):
    return con.execute(query).fetchdf()


---
## Exercise 1 · Month-over-Month Sales Comparison

**Question:** For each calendar month, calculate the **total sales amount** and compare it to the **previous month's total** using `LAG`. Show `month_label`, `monthly_total`, `prev_month_total`, and the **absolute difference**. Include all months from 2022–2023.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    month_label,
    monthly_total,
    LAG(monthly_total) OVER (ORDER BY month_label) AS prev_month_total,
    monthly_total - LAG(monthly_total) OVER (ORDER BY month_label) AS mom_diff
FROM (
    SELECT month_label, SUM(amount) AS monthly_total
    FROM sales
    GROUP BY month_label
)
ORDER BY month_label;
```
</details>

---
## Exercise 2 · Month-over-Month Growth Rate (%)

**Question:** Extend Exercise 1 to calculate the **MoM growth rate** as a percentage, rounded to 2 decimals. A positive value means revenue grew; negative means it shrank. Exclude the first month (no prior data).

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    month_label,
    monthly_total,
    prev_month_total,
    ROUND(
        (monthly_total - prev_month_total) / prev_month_total * 100, 2
    ) AS mom_growth_pct
FROM (
    SELECT
        month_label,
        SUM(amount) AS monthly_total,
        LAG(SUM(amount)) OVER (ORDER BY month_label) AS prev_month_total
    FROM sales
    GROUP BY month_label
)
WHERE prev_month_total IS NOT NULL
ORDER BY month_label;
```
</details>

---
## Exercise 3 · Each Sale vs. Previous Sale (per Rep)

**Question:** For each sale, show the **amount of the previous sale by the same rep** (`prev_sale_amount`). Include `rep_name`, `sale_date`, `amount`, `prev_sale_amount`. Use 0 for the first sale.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    rep_name,
    sale_date,
    amount,
    LAG(amount, 1, 0) OVER (
        PARTITION BY rep_name
        ORDER BY sale_date
    ) AS prev_sale_amount
FROM sales
ORDER BY rep_name, sale_date;
```
</details>

---
## Exercise 4 · Was the Last Sale Their Best?

**Question:** For each sales rep, find out whether their **most recent sale** (by date) was **their highest sale ever**. Return `rep_name`, `last_sale_amount`, `max_ever_amount`, and a boolean column `ended_on_peak`.

*Tip: Use `LAST_VALUE` with an unbounded frame, or use a subquery with `MAX`.*

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
WITH rep_stats AS (
    SELECT
        rep_name,
        amount,
        MAX(amount) OVER (PARTITION BY rep_name) AS max_ever_amount,
        LAST_VALUE(amount) OVER (
            PARTITION BY rep_name
            ORDER BY sale_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
        ) AS last_sale_amount
    FROM sales
)
SELECT DISTINCT
    rep_name,
    last_sale_amount,
    max_ever_amount,
    last_sale_amount = max_ever_amount AS ended_on_peak
FROM rep_stats
ORDER BY rep_name;
```
</details>

---
## Exercise 5 · Variance from Career-First Sale

**Question:** For each sale, show the **difference from that rep's very first sale amount**. A positive delta means the sale was larger than their debut. Show `rep_name`, `sale_date`, `amount`, `first_sale_amount`, `delta_from_first`.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    rep_name,
    sale_date,
    amount,
    FIRST_VALUE(amount) OVER (
        PARTITION BY rep_name
        ORDER BY sale_date
    ) AS first_sale_amount,
    amount - FIRST_VALUE(amount) OVER (
        PARTITION BY rep_name
        ORDER BY sale_date
    ) AS delta_from_first
FROM sales
ORDER BY rep_name, sale_date;
```
</details>

---
## Exercise 6 · Preview the Next Sale (LEAD)

**Question:** For each sale, show the **date of the next sale by the same rep** and the **number of days until that next sale**. If there is no next sale, return `NULL`. Show `rep_name`, `sale_date`, `next_sale_date`, `days_until_next_sale`.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    rep_name,
    sale_date,
    LEAD(sale_date) OVER (
        PARTITION BY rep_name
        ORDER BY sale_date
    ) AS next_sale_date,
    DATEDIFF('day',
        CAST(sale_date AS DATE),
        CAST(LEAD(sale_date) OVER (PARTITION BY rep_name ORDER BY sale_date) AS DATE)
    ) AS days_until_next_sale
FROM sales
ORDER BY rep_name, sale_date;
```
</details>

---
## Exercise 7 · Third Sale of Each Rep (NTH_VALUE)

**Question:** Use `NTH_VALUE` to find each rep's **3rd sale amount** (by date). Return one row per rep with `rep_name` and `third_sale_amount`. Reps with fewer than 3 sales should show `NULL`.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT DISTINCT
    rep_name,
    NTH_VALUE(amount, 3) OVER (
        PARTITION BY rep_name
        ORDER BY sale_date
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING
    ) AS third_sale_amount
FROM sales
ORDER BY rep_name;
```
</details>

---
## Exercise 8 · Hiring Gap Between Teammates

**Question:** Within each department, show the **number of days between each employee's hire date and the hire date of the person hired just before them** (ordered by hire date). Show `full_name`, `department`, `hire_date`, `prev_hire_date`, `days_since_last_hire`.

In [ ]:
# Your answer here


<details><summary>💡 Solution</summary>

```sql
SELECT
    full_name,
    department,
    hire_date,
    LAG(hire_date) OVER (
        PARTITION BY department
        ORDER BY hire_date
    ) AS prev_hire_date,
    DATEDIFF('day',
        CAST(LAG(hire_date) OVER (PARTITION BY department ORDER BY hire_date) AS DATE),
        CAST(hire_date AS DATE)
    ) AS days_since_last_hire
FROM employees
ORDER BY department, hire_date;
```
</details>

---
## 🎯 Summary

- `LAG(col, n, default)` and `LEAD(col, n, default)` → compare rows across time; essential for period-over-period metrics.
- `FIRST_VALUE(col)` / `LAST_VALUE(col)` → anchor comparisons to the start or end of a group.
- `NTH_VALUE(col, n)` → grab any specific position in a ranked set.
- Always define `ORDER BY` inside `OVER()` for offset functions.
- For `LAST_VALUE`, expand the frame: `ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING`.

➡️ **Next:** `03_running_aggregates.ipynb` — SUM OVER, rolling averages, cumulative metrics